In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch

from collections import Counter
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

# -----------------------------
# 0. ENVIRONMENT SETUP
# -----------------------------
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# 1. LOAD DATA
# -----------------------------
df = pd.read_csv("review.csv")
df = df.rename(columns={"review_text": "text"})
df = df.dropna(subset=["text", "rating"])

def create_label(row):
    if row["rating"] >= 4:
        return 1   # positive
    elif row["rating"] <= 2:
        return 0   # negative
    else:
        return 2   # neutral/sarcastic

df["label"] = df.apply(create_label, axis=1)
df = df[["text", "label"]]

print("\nDataset preview:")
print(df.head())

print("\nClass distribution:")
print(df["label"].value_counts())

# -----------------------------
# 2. TRAIN / VALIDATION SPLIT
# -----------------------------
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

print(f"\nTraining samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# -----------------------------
# 3. TOKENIZER
# -----------------------------
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------------
# 4. CLASS WEIGHTS
# -----------------------------
label_counts = Counter(train_df["label"].tolist())
total = len(train_df)
num_classes = 3

class_weights = torch.tensor(
    [total / (num_classes * label_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
)

print("\nClass weights:")
print(class_weights)

# -----------------------------
# 5. METRICS
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# -----------------------------
# 6. CUSTOM TRAINER
# -----------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# -----------------------------
# 7. LOAD MODEL
# -----------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
)

# Partial freezing: freeze embeddings + first 6 encoder layers
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False

for layer in model.roberta.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")

# -----------------------------
# 8. TRAINING ARGUMENTS
# -----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False
)

# -----------------------------
# 9. TRAINER
# -----------------------------
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

# -----------------------------
# 10. TRAIN
# -----------------------------
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)

trainer.train()

# -----------------------------
# 11. FINAL EVALUATION
# -----------------------------
print("\nFinal evaluation on validation set:")
eval_results = trainer.evaluate()
print(eval_results)

print(f"\nValidation Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Validation Precision: {eval_results['eval_precision']:.4f}")
print(f"Validation Recall: {eval_results['eval_recall']:.4f}")
print(f"Validation F1: {eval_results['eval_f1']:.4f}")

pred_output = trainer.predict(val_dataset)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

label_names = ["negative", "positive", "neutral/sarcastic"]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

# -----------------------------
# 12. SAVE MODEL
# -----------------------------
save_path = "sentiment_sarcasm_model"
os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

label_mapping = {
    0: "negative",
    1: "positive",
    2: "neutral/sarcastic"
}

with open(os.path.join(save_path, "label_mapping.json"), "w") as f:
    json.dump(label_mapping, f, indent=4)

with open(os.path.join(save_path, "eval_results.json"), "w") as f:
    json.dump(eval_results, f, indent=4)

print(f"\n✅ Training complete. Best model saved to '{save_path}'")
print("✅ Accuracy, Precision, Recall, and F1 are shown after each epoch and in final evaluation.")

c:\Users\nazeem\OneDrive\Desktop\hackculture2.0\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu

Dataset preview:
                                                text  label
0  Excellent phone! Camera quality is top notch 📸...      1
1  Phone acha hai but price thoda zyada hai. Came...      1
2  ಬಹಳ ಚೆನ್ನಾಗಿದೆ! Camera super aagide. Battery b...      1
3  worst phone ever!!!!! hangs every 5 minuts. do...      0
4  Heating issue is terrible. Phone becomes like ...      0

Class distribution:
label
0    707
1    491
2     16
Name: count, dtype: int64

Training samples: 1092
Validation samples: 122


Map: 100%|██████████| 122/122 [00:00<00:00, 23333.57 examples/s]



Class weights:
tensor([ 0.5723,  0.8235, 26.0000])


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4457.43it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

{'eval_loss': 0.14498569071292877, 'eval_accuracy': 0.9836065573770492, 'eval_precision': 0.9918032786885246, 'eval_recall': 0.9836065573770492, 'eval_f1': 0.9861680327868851}

Validation Accuracy: 0.9836
Validation Precision: 0.9918
Validation Recall: 0.9836
Validation F1: 0.9862


c:\Users\nazeem\OneDrive\Desktop\hackculture2.0\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Classification Report:
                   precision    recall  f1-score   support

         negative       1.00      1.00      1.00        71
         positive       1.00      0.96      0.98        49
neutral/sarcastic       0.50      1.00      0.67         2

         accuracy                           0.98       122
        macro avg       0.83      0.99      0.88       122
     weighted avg       0.99      0.98      0.99       122



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.38it/s]


✅ Training complete. Best model saved to 'sentiment_sarcasm_model'
✅ Accuracy, Precision, Recall, and F1 are shown after each epoch and in final evaluation.


In [2]:
print(df.columns)

Index(['product_id', 'product_name', 'category', 'review_text', 'rating',
       'timestamp', 'reviewer_id', 'language_type'],
      dtype='str')


In [1]:
import accelerate
print("✅ Accelerate installed:", accelerate.__version__)

✅ Accelerate installed: 1.13.0


c:\Users\nazeem\OneDrive\Desktop\hackculture2.0\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
